In [ ]:
"""
Verarbeitung von BSI/CERT-Bund-Bedrohungsinformationen.

Die Meldungen werden zunächst über den RSS-Feed eingelesen,
mit den verfügbaren CSAF-Daten angereichert und anschließend
als STIX 2.1 für OpenCTI aufbereitet.

Benötigte Pakete:
    pip install feedparser requests
"""

import json
import re
import uuid
from datetime import datetime, timezone
from email.utils import parsedate_to_datetime
from pathlib import Path

import feedparser
import requests
import truststore

# Zertifikate ueber die Zertifikatsverwaltung des Betriebssystems prüfen.
# Python lädt fehlende Zwischenzertifikate nicht selbst nach, Browser schon.
truststore.inject_into_ssl()

# --------------------------------------------------------------------------
# Einstellungen
# --------------------------------------------------------------------------

FEED_URL = "https://wid.cert-bund.de/content/public/securityAdvisory/rss"
CSAF_URL_TEMPLATE = "https://wid.cert-bund.de/.well-known/csaf/white/{year}/wid-sec-w-{year}-{num}.json"

DATA_DIR = Path("data")
RAW_DIR = DATA_DIR / "raw"          # Rohdaten: RSS-Eintraege + CSAF-JSON 
BUNDLE_DIR = DATA_DIR / "bundles"   # erzeugte STIX-Bundles 
STATE_FILE = DATA_DIR / "state.json"  # Meldungsbestand 

# fester Namensraum für deterministische STIX-Kennungen 
NAMESPACE = uuid.UUID("6ba7b810-9dad-11d1-80b4-00c04fd430c8")

SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "bsi-cti-prototyp/0.1 (Bachelorarbeit)"})

# --------------------------------------------------------------------------
# Hilfsfunktionen
# --------------------------------------------------------------------------

def now_iso() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.000Z")


def to_iso(date: str | None):
    #Wandelt das Feed-Datum in einen ISO-Zeitstempel um
    try:
        return parsedate_to_datetime(date).astimezone(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.000Z")
    except (TypeError, ValueError):
        return None


def to_stix_ts(ts: str | None):
    #Wandelt einen ISO-Zeitstempel in das STIX-Format um
    if not ts:
        return None
    try:
        dt = datetime.fromisoformat(ts.strip().replace("Z", "+00:00"))
        if dt.tzinfo is None:
            dt = dt.replace(tzinfo=timezone.utc)
        return dt.astimezone(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.000Z")
    except ValueError:
        return None


def normalize_cpe(cpe: str | None):
    #bringt CPE-Angaben in das von STIX erwartete Format
    if not cpe:
        return None
    cpe = cpe.strip()
    if not cpe.startswith("cpe:/"):
        return cpe                                      
    parts = (cpe[5:].split(":") + [""] * 7)[:7]         
    part, vendor, product, version, update, edition, language = parts
    sw_edition = target_sw = target_hw = other = ""
    if edition.startswith("~"):  
        edition, sw_edition, target_sw, target_hw, other = (edition.split("~") + [""] * 6)[1:6]
    comps = [part, vendor, product, version, update, edition, language,
             sw_edition, target_sw, target_hw, other]
    return "cpe:2.3:" + ":".join(c if c != "" else "*" for c in comps)


def prune(value):
    #entfernt leere Werte aus dem STIX-Objekt
    if isinstance(value, dict):
        out = {k: prune(v) for k, v in value.items()}
        return {k: v for k, v in out.items() if v not in (None, "", [], {})}
    if isinstance(value, list):
        out = [prune(v) for v in value]
        return [v for v in out if v not in (None, "", [], {})]
    return value


def stix_id(object_type: str, key: str):
    #erzeugt eine deterministische STIX-ID
    return f"{object_type}--{uuid.uuid5(NAMESPACE, f'{object_type}:{key}')}"


def ensure_dirs():
    for d in (RAW_DIR, BUNDLE_DIR):
        d.mkdir(parents=True, exist_ok=True)


# --------------------------------------------------------------------------
# Stufe 1: Erfassung
# --------------------------------------------------------------------------

# Tolerante Titelkonvention: "[NEU] [mittel] Produkt: Beschreibung"
TITLE_RE = re.compile(r"^\[(?P<status>NEU|UPDATE)\]\s*\[(?P<severity>niedrig|mittel|hoch|kritisch)\]\s*(?P<rest>.*)$", re.IGNORECASE)
WID_RE = re.compile(r"name=(WID-SEC-\d{4}-\d+)")


def parse_title(title: str):
    # liest Status und Schweregrad aus dem Meldungstitel
    m = TITLE_RE.match(title or "")
    if not m:
        return {"status": None, "severity": None}
    return {"status": m.group("status").upper(), "severity": m.group("severity").lower()}


def fetch_feed(url: str = FEED_URL):
    """RSS-Feed abrufen und WID-Kennungen extrahieren."""
    r = SESSION.get(url, timeout=30)
    r.raise_for_status()
    feed = feedparser.parse(r.content)
    if feed.bozo and not feed.entries:
        raise RuntimeError(f"Feed nicht parsebar: {feed.bozo_exception}")
    entries = []
    for e in feed.entries:
        m = WID_RE.search(e.get("link", ""))
        if not m:
            continue  # Einträge ohne Kennung koennen nicht verarbeitet werden
        entries.append({
            "wid_id": m.group(1),
            "title": e.get("title", ""),
            "summary": e.get("summary", ""),
            "link": e.get("link", ""),
            "published": e.get("published", ""),
            **parse_title(e.get("title", "")),
        })
    return entries


# --------------------------------------------------------------------------
# Stufe 2 : Zustandsverwaltung
# --------------------------------------------------------------------------
def load_state():
    if STATE_FILE.exists():
        return json.loads(STATE_FILE.read_text(encoding="utf-8"))
    return {}


def save_state(state: dict):
    STATE_FILE.write_text(json.dumps(state, indent=2, ensure_ascii=False), encoding="utf-8")


def needs_processing(entry: dict, state: dict):
    #prüft, ob eine Meldung verarbeitet werden muss
    known = state.get(entry["wid_id"])
    if known is None:
        return True
    return entry.get("status") == "UPDATE"


# --------------------------------------------------------------------------
# Stufe 3: Anreicherung
# --------------------------------------------------------------------------

def csaf_url(wid_id: str):
    _, _, year, num = wid_id.split("-")
    return CSAF_URL_TEMPLATE.format(year=year, num=num).lower().replace("https", "https", 1)


def fetch_csaf(wid_id: str):
    # ruft das zugehörige CSAF-Dokument ab
    try:
        r = SESSION.get(csaf_url(wid_id), timeout=30)
        r.raise_for_status()
        return r.json()
    except Exception as exc:
        print(f"  [WARN] CSAF-Abruf fehlgeschlagen fuer {wid_id}: {exc}")
        return None


def get_note(doc: dict, category: str, title: str | None = None):
    #liest eine Note aus dem CSAF-Dokument
    for note in doc.get("document", {}).get("notes", []):
        if note.get("category") != category:
            continue
        if title and note.get("title") != title:
            continue
        return note.get("text")
    return None


def collect_products(doc: dict):
    #liest die Produkte aus dem CSAF-Produktbaum und sammelt diezugehörigen Produktinfos
    products: dict[str, dict] = {}

    def add(product: dict | None, vendor: str | None, category: str | None) -> None:
        if not product:
            return
        pid = product.get("product_id")
        if not pid:
            return
        products[pid] = {
            "product_id": pid,
            "name": product.get("name"),
            "vendor": vendor,
            "cpe": normalize_cpe((product.get("product_identification_helper") or {}).get("cpe")),
            "category": category,
            "cpe_derived": False,
        }

    def walk(branches: list | None, vendor: str | None):
        for b in branches or []:
            category = b.get("category")
            v = b.get("name") if category == "vendor" else vendor
            add(b.get("product"), v, category)
            walk(b.get("branches"), v)

    tree = doc.get("product_tree") or {}
    walk(tree.get("branches"), None)
    for fp in tree.get("full_product_names") or []:
        add(fp, None, "full_product_name")
    for rel in tree.get("relationships") or []:
        add(rel.get("full_product_name"), None, "relationship")
    return products


def classify_products(doc, products):
    #ordnet die Produkte nach ihrem Status ein
    affected_ids: set = set()
    fixed_ids: set = set()
    for v in doc.get("vulnerabilities") or []:
        ps = v.get("product_status") or {}
        for key in ("known_affected", "last_affected"):
            affected_ids.update(ps.get(key) or [])
        for key in ("fixed", "first_fixed"):
            fixed_ids.update(ps.get(key) or [])

    if not affected_ids:  # Rückfall: Branch-Kategorie
        affected_ids = {p["product_id"] for p in products.values()
                        if p["category"] == "product_version_range"}
        if not fixed_ids:
            fixed_ids = {p["product_id"] for p in products.values()
                         if p["category"] == "product_version"}

    # CPE-Uebertragung von behobener auf betroffene Version desselben
    # Produktzweigs, als abgeleitetes Merkmal gekennzeichnet (A6)
    for pid in affected_ids:
        p = products.get(pid)
        if not p or p["cpe"]:
            continue
        for fid in fixed_ids:
            f = products.get(fid)
            if f and f["cpe"] and fid.startswith(f"{pid}-"):
                p["cpe"] = f["cpe"]
                p["cpe_derived"] = True
                break
    return affected_ids, fixed_ids


def extract_csaf(doc: dict):
    #extrahiert die relevanten Daten aus dem CSAF-Dokument
    d = doc.get("document", {})
    tracking = d.get("tracking", {})
    products = collect_products(doc)
    affected_ids, fixed_ids = classify_products(doc, products)

    vulns = []
    for v in doc.get("vulnerabilities") or []:
        if not v.get("cve"):
            continue
        ps = v.get("product_status") or {}
        ids = list((ps.get("known_affected") or [])) + list((ps.get("last_affected") or []))
        vulns.append({"cve": v["cve"], "affected_ids": ids})

    references = [
        {"summary": r.get("summary"), "url": r.get("url")}
        for r in d.get("references", [])
        if r.get("category") == "external"
    ]
    return {
        "title": d.get("title"),
        "tracking_id": tracking.get("id"),
        "version": tracking.get("version"),
        "initial_release": tracking.get("initial_release_date"),
        "revision_history": tracking.get("revision_history", []),
        "aggregate_severity": (d.get("aggregate_severity") or {}).get("text"),
        "summary": get_note(doc, "summary"),
        "product_description": get_note(doc, "description"),
        "operating_systems": get_note(doc, "general", title="Betroffene Betriebssysteme"),
        "references": references,
        "products": products,
        "affected_ids": affected_ids,
        "fixed_ids": fixed_ids,
        "vulnerabilities": vulns,
    }


# --------------------------------------------------------------------------
# Stufe 4: STIX-Abbildung 
# --------------------------------------------------------------------------

def make_identity():
    return {
        "type": "identity",
        "spec_version": "2.1",
        "id": stix_id("identity", "CERT-Bund"),
        "created": now_iso(),
        "modified": now_iso(),
        "name": "CERT-Bund (BSI)",
        "identity_class": "organization",
    }


def make_report_base(entry: dict, identity_id: str):
    #Erzeugt den Basis-Report aus den Feed-Daten
    return {
        "type": "report",
        "spec_version": "2.1",
        "id": stix_id("report", entry["wid_id"]),
        "created_by_ref": identity_id,
        "created": now_iso(),
        "modified": now_iso(),
        "name": entry["title"],
        "description": entry["summary"],
        "published": to_iso(entry.get("published")) or now_iso(),
        "report_types": ["vulnerability"],
        "object_refs": [identity_id],
        "external_references": [
            {"source_name": "WID-Portal", "external_id": entry["wid_id"], "url": entry["link"]},
        ],
        "x_wid_status": entry.get("status"),
        "x_wid_severity": entry.get("severity"),
    }


def make_enriched(entry: dict, csaf: dict, identity_id: str):
    #Erzeugt den STIX-Report mit den zusätzlichen CSAF-Daten
    objects: list[dict] = []
    products = csaf["products"]

    # Pro Produkt wird ein Software-Objekt angelegt.
#    gleiche Software wird über die ID nur einmal übernommen
    sw_by_product_id: dict[str, str] = {}
    seen_sw: set = set()
    for pid in csaf["affected_ids"]:
        prod = products.get(pid)
        if not prod:
            continue
        sw_id = stix_id("software", prod["name"] or pid)
        sw_by_product_id[pid] = sw_id
        if sw_id in seen_sw:
            continue
        seen_sw.add(sw_id)
        sw = {
            "type": "software",
            "spec_version": "2.1",
            "id": sw_id,
            "name": prod["name"] or pid,
        }
        if prod.get("vendor"):
            sw["vendor"] = prod["vendor"]
        if prod.get("cpe"):
            sw["cpe"] = prod["cpe"]
            if prod.get("cpe_derived"):
                sw["x_cpe_derived_from_fixed_version"] = True
        objects.append(sw)

    # CVEs und Beziehungen zu den betroffenen Produkten
    seen_rel: set = set()
    for v in csaf["vulnerabilities"]:
        vuln = {
            "type": "vulnerability",
            "spec_version": "2.1",
            "id": stix_id("vulnerability", v["cve"]),
            "created": now_iso(),
            "modified": now_iso(),
            "name": v["cve"],
            "external_references": [{"source_name": "cve", "external_id": v["cve"]}],
        }
        objects.append(vuln)
        for pid in v["affected_ids"]:
            sw_id = sw_by_product_id.get(pid)
            if not sw_id:
                continue
            rel_id = stix_id("relationship", f"{sw_id}:has:{vuln['id']}")
            if rel_id in seen_rel:
                continue
            seen_rel.add(rel_id)
            objects.append({
                "type": "relationship",
                "spec_version": "2.1",
                "id": rel_id,
                "created": now_iso(),
                "modified": now_iso(),
                "relationship_type": "has",
                "source_ref": sw_id,
                "target_ref": vuln["id"],
            })

    # behobene Versionen als Gegenmassnahmen-Information
    fixed_names = sorted({products[pid]["name"] for pid in csaf["fixed_ids"]
                          if pid in products and products[pid]["name"]})

    ext_refs = [
        {"source_name": "WID-Portal", "external_id": entry["wid_id"], "url": entry["link"]},
        {"source_name": "CSAF", "external_id": csaf.get("tracking_id"), "url": csaf_url(entry["wid_id"])},
    ]
    for ref in csaf["references"]:
        ext_refs.append({"source_name": ref.get("summary") or "extern", "url": ref.get("url")})

    report = {
        "type": "report",
        "spec_version": "2.1",
        "id": stix_id("report", f"{entry['wid_id']}-enriched"),
        "created_by_ref": identity_id,
        "created": now_iso(),
        "modified": now_iso(),
        "name": csaf.get("title") or entry["title"],
        "description": csaf.get("summary") or entry["summary"],
        "published": to_stix_ts(csaf.get("initial_release")) or now_iso(),
        "report_types": ["vulnerability"],
        "object_refs": [identity_id] + [o["id"] for o in objects],
        "external_references": ext_refs,
        "x_wid_status": entry.get("status"),
        "x_wid_severJity": csaf.get("aggregate_severity") or entry.get("severity"),
        "x_wid_revision": csaf.get("version"),
        "x_operating_systems": csaf.get("operating_systems"),
        "x_product_description": csaf.get("product_description"),
        "x_fixed_version": ", ".join(fixed_names) or None,
    }
    return [report] + objects


def make_bundle(objects: list[dict]) -> dict:
    objects = [prune(o) for o in objects if o]
    return {"type": "bundle", "id": f"bundle--{uuid.uuid4()}", "objects": objects}


# --------------------------------------------------------------------------
# Stufe 5 : Ausgabe und Rohdaten schreiben
# --------------------------------------------------------------------------

def persist_raw(wid_id: str, entry: dict, csaf_doc: dict | None):
    #speichert die Rohdaten zusammen mit dem Abrufzeitpunkt
    stamp = {"fetched_at": now_iso()}
    (RAW_DIR / f"{wid_id}_rss.json").write_text(
        json.dumps({**stamp, "entry": entry}, indent=2, ensure_ascii=False), encoding="utf-8")
    if csaf_doc is not None:
        (RAW_DIR / f"{wid_id}_csaf.json").write_text(
            json.dumps(csaf_doc, indent=2, ensure_ascii=False), encoding="utf-8")


def process_entry(entry: dict, identity: dict):
    #verarbeitet eine Meldung und erstellt die beiden Bundle-Varianten
    wid_id = entry["wid_id"]
    csaf_doc = fetch_csaf(wid_id)  # A4
    persist_raw(wid_id, entry, csaf_doc)  # A9

    base_bundle = make_bundle([identity, make_report_base(entry, identity["id"])])
    (BUNDLE_DIR / f"{wid_id}_base.json").write_text(
        json.dumps(base_bundle, indent=2, ensure_ascii=False), encoding="utf-8")

    stats = {"wid_id": wid_id, "status": entry.get("status"),
             "severity": entry.get("severity"),
             "base_objects": len(base_bundle["objects"]),
             "enriched_objects": None, "cves": 0, "software": 0, "relationships": 0}

    if csaf_doc is not None:
        csaf = extract_csaf(csaf_doc)  
        enriched = make_enriched(entry, csaf, identity["id"])
        bundle = make_bundle([identity] + enriched)
        (BUNDLE_DIR / f"{wid_id}_enriched.json").write_text(
            json.dumps(bundle, indent=2, ensure_ascii=False), encoding="utf-8")
        stats["enriched_objects"] = len(bundle["objects"])
        stats["cves"] = sum(1 for o in enriched if o["type"] == "vulnerability")
        stats["software"] = sum(1 for o in enriched if o["type"] == "software")
        stats["relationships"] = sum(1 for o in enriched if o["type"] == "relationship")
    return stats


STATS_FIELDS = ["run_at", "wid_id", "status", "severity", "base_objects",
                "enriched_objects", "cves", "software", "relationships"]


def append_stats(rows: list[dict]):
    #hängt die Verarbeitungskennzahlen an die Statistikdatei an
    stats_file = DATA_DIR / "stats.csv"
    new_file = not stats_file.exists()
    run_at = now_iso()
    with stats_file.open("a", encoding="utf-8") as f:
        if new_file:
            f.write(";".join(STATS_FIELDS) + "\n")
        for row in rows:
            values = [run_at] + [str(row.get(k, "") if row.get(k) is not None else "")
                                 for k in STATS_FIELDS[1:]]
            f.write(";".join(values) + "\n")


def run(limit: int | None = None):
    ensure_dirs()                                                  # Ordner anlegen
    state = load_state()                                           # Meldungsbestand laden 
    identity = make_identity()                                      # BSI als Herausgeber, einmal für alle
    entries = fetch_feed()                                          # Feed abrufen 
    print(f"{len(entries)} Feed-Eintraege gefunden.")

    processed = 0
    all_stats: list[dict] = []
    for entry in entries:
        if limit and processed >= limit:
            break
        if not needs_processing(entry, state): 
            continue
        print(f"Verarbeite {entry['wid_id']} ({entry.get('status')}, {entry.get('severity')}) ...")
        stats = process_entry(entry, identity)
        all_stats.append(stats)
        print(f"  Basis: {stats['base_objects']} Objekte | angereichert: "
              f"{stats['enriched_objects']} Objekte, {stats['cves']} CVEs, "
              f"{stats['software']} Software, {stats['relationships']} Beziehungen")
        state[entry["wid_id"]] = {"processed_at": now_iso(), "status": entry.get("status")}
        processed += 1

    save_state(state)  
    append_stats(all_stats)
    print(f"Fertig: {processed} Meldungen verarbeitet. Kennzahlen: data/stats.csv")


# Achtung: run() ruft den Feed live ab und haengt Zeilen an stats.csv an
# beim erneuten Ausführen dieser Zelle (z. B. nach Aenderungen) auf False lassen
RUN_PIPELINE = False
if RUN_PIPELINE:
    run(limit=None)

## Bundles aus den Rohdaten neu erzeugen

Erzeugt die Bundles des Laufs vom 24.09.2026 erneut, **ohne Netzzugriff**, aus den in `data/raw/` gesicherten
RSS-Einträgen und CSAF-Dokumenten (A9). Die Ausgabe landet in `data/bundles_final/`, sodass dort genau
250 × 2 Bundles liegen und keine Dateien früherer Läufe.

Die Gegenprobe vergleicht die Objektzahlen jeder Meldung mit der `stats.csv` des Laufs.

In [2]:
import csv

# 1. Meldungen des jüngsten Laufs aus stats.csv
with (DATA_DIR / "stats.csv").open(encoding="utf-8") as f:
    rows = list(csv.DictReader(f, delimiter=";"))
latest = max(r["run_at"] for r in rows)
lauf = [r for r in rows if r["run_at"] == latest]
print(f"Lauf {latest}: {len(lauf)} Meldungen")

OUT_DIR = DATA_DIR / "bundles_final"
OUT_DIR.mkdir(parents=True, exist_ok=True)
for alt in OUT_DIR.glob("*.json"):   # sauberer Zustand bei Wiederholung
    alt.unlink()

def schreibe(pfad, bundle):
    pfad.write_text(json.dumps(bundle, indent=2, ensure_ascii=False), encoding="utf-8")

identity = make_identity()
abweichungen, fehlende_rohdaten = [], []

for r in lauf:
    wid = r["wid_id"]
    rss_pfad = RAW_DIR / f"{wid}_rss.json"
    if not rss_pfad.exists():
        fehlende_rohdaten.append(wid)
        continue
    entry = json.loads(rss_pfad.read_text(encoding="utf-8"))["entry"]

    schreibe(OUT_DIR / f"{wid}_base.json",
             make_bundle([identity, make_report_base(entry, identity["id"])]))

    csaf_pfad = RAW_DIR / f"{wid}_csaf.json"
    if csaf_pfad.exists():
        csaf = extract_csaf(json.loads(csaf_pfad.read_text(encoding="utf-8")))
        enriched = make_enriched(entry, csaf, identity["id"])
        bundle = make_bundle([identity] + enriched)
        schreibe(OUT_DIR / f"{wid}_enriched.json", bundle)

        ist = (len(bundle["objects"]),
               sum(o["type"] == "vulnerability" for o in bundle["objects"]),
               sum(o["type"] == "software" for o in bundle["objects"]),
               sum(o["type"] == "relationship" for o in bundle["objects"]))
        soll = tuple(int(r[k] or 0) for k in ("enriched_objects", "cves", "software", "relationships"))
        if ist != soll:
            abweichungen.append((wid, ist, soll))

n_base = len(list(OUT_DIR.glob("*_base.json")))
n_enr = len(list(OUT_DIR.glob("*_enriched.json")))
print(f"Geschrieben: {n_base} Basis- und {n_enr} angereicherte Bundles in {OUT_DIR}")
print(f"Fehlende Rohdaten: {fehlende_rohdaten or 'keine'}")
print(f"Abweichungen gegenueber stats.csv: {len(abweichungen)}")
for w in abweichungen[:10]:
    print("  ", w)

Lauf 2026-09-24T17:32:36.000Z: 250 Meldungen
Geschrieben: 250 Basis- und 250 angereicherte Bundles in data/bundles_final
Fehlende Rohdaten: keine
Abweichungen gegenueber stats.csv: 0
